In [0]:
%run ../_shared/00_project_config

In [0]:
%run ../_shared/01_spark_utils

In [0]:
from pyspark.sql import functions as F


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
# Project-wide schema names and supported FBref leagues are loaded from
# 00_project_config.
#
# Reusable reading, cleaning, validation, deduplication, and writing helpers
# are loaded from 01_spark_utils.

SILVER_TABLE = f"{SILVER_SCHEMA}.team_playing_time_stats"

FBREF_DATASET = "playing_time"

# One Silver row represents:
#
#   team + season + league
#
# for the FBref playing-time dataset.
BUSINESS_KEY = [
    "team_name",
    "season_code",
    "league_code",
]

REQUIRED_SILVER_COLUMNS = [
    "team_name",
    "team_key",
    "season_code",
    "season_start_year",
    "league_code",
]


# -----------------------------------------------------------------------------
# Build expected Bronze table list
# -----------------------------------------------------------------------------
# FBref Bronze maintains one table per:
#
#   dataset + league
#
# Each Bronze table contains all supported seasons for that league.

BRONZE_TABLES = [
    f"{BRONZE_SCHEMA}.fbref_{FBREF_DATASET}_{league_code}"
    for league_code in FBREF_LEAGUES
]


# -----------------------------------------------------------------------------
# Read and combine Bronze tables
# -----------------------------------------------------------------------------
# read_and_union_tables() is strict by default:
#
#   - every expected league table must exist
#   - schemas must be compatible
#
# A missing league should fail the pipeline rather than silently produce an
# incomplete Silver dataset.

bronze_df = read_and_union_tables(
    spark,
    BRONZE_TABLES,
)


# -----------------------------------------------------------------------------
# Validate expected FBref source columns
# -----------------------------------------------------------------------------
# FBref is an external HTML source and its table structure may change.
# Validate every source field used by this transformation before selecting it.

REQUIRED_BRONZE_COLUMNS = [
    "Squad",
    "#_Pl",
    "Age",
    "Playing_Time_MP",
    "Playing_Time_Min",
    "Playing_Time_Mn/MP",
    "Playing_Time_Min%",
    "Playing_Time_90s",
    "Starts_Starts",
    "Starts_Mn/Start",
    "Starts_Compl",
    "Subs_Subs",
    "Subs_Mn/Sub",
    "Subs_unSub",
    "Team_Success_PPM",
    "Team_Success_onG",
    "Team_Success_onGA",
    "Team_Success_+/-",
    "Team_Success_+/-90",
    "_season",
    "_league",
    "_stat",
    "_ingested_at",
    "_source_file",
    "_source_system",
]

validate_columns_exist(
    bronze_df,
    REQUIRED_BRONZE_COLUMNS,
)


# -----------------------------------------------------------------------------
# Clean top-level strings
# -----------------------------------------------------------------------------
# FBref Bronze is flat, so the shared top-level string cleaner is appropriate.
#
# It trims string values and converts empty strings to NULL.

bronze_df = clean_top_level_strings(bronze_df)


# -----------------------------------------------------------------------------
# Validate dataset identity
# -----------------------------------------------------------------------------
# All rows processed in this notebook should originate from the FBref
# "playing_time" dataset.

unexpected_stat_rows = (
    bronze_df
    .filter(
        F.col("_stat").isNull()
        | (F.col("_stat") != F.lit(FBREF_DATASET))
    )
    .limit(1)
    .count()
)

if unexpected_stat_rows > 0:
    raise ValueError(
        f"Unexpected _stat value found while processing {FBREF_DATASET}."
    )


# -----------------------------------------------------------------------------
# Select, rename, and cast Silver columns
# -----------------------------------------------------------------------------
# Source-oriented FBref field names are converted into clearer analytical
# names.
#
# Unlike the original notebook, useful playing-time measures are retained
# rather than discarded. These fields provide valuable context for downstream
# analysis and Gold-layer metrics.

silver_df = (
    bronze_df
    .select(

        # ---------------------------------------------------------------------
        # Team
        # ---------------------------------------------------------------------

        clean_string("Squad")
        .alias("team_name"),

        F.col("`#_Pl`")
        .cast("int")
        .alias("players_used_count"),

        F.col("Age")
        .cast("double")
        .alias("average_age"),


        # ---------------------------------------------------------------------
        # Playing time
        # ---------------------------------------------------------------------

        F.col("Playing_Time_MP")
        .cast("int")
        .alias("matches_played"),

        F.col("Playing_Time_Min")
        .cast("int")
        .alias("minutes_played"),

        F.col("`Playing_Time_Mn/MP`")
        .cast("double")
        .alias("minutes_per_match"),

        F.col("`Playing_Time_Min%`")
        .cast("double")
        .alias("minutes_percentage"),

        F.col("Playing_Time_90s")
        .cast("double")
        .alias("nineties_played"),


        # ---------------------------------------------------------------------
        # Starts
        # ---------------------------------------------------------------------

        F.col("Starts_Starts")
        .cast("int")
        .alias("starts"),

        F.col("`Starts_Mn/Start`")
        .cast("double")
        .alias("minutes_per_start"),

        F.col("Starts_Compl")
        .cast("int")
        .alias("complete_matches_started"),


        # ---------------------------------------------------------------------
        # Substitutions
        # ---------------------------------------------------------------------

        F.col("Subs_Subs")
        .cast("int")
        .alias("substitute_appearances"),

        F.col("`Subs_Mn/Sub`")
        .cast("double")
        .alias("minutes_per_substitute_appearance"),

        F.col("Subs_unSub")
        .cast("int")
        .alias("unused_substitute_appearances"),


        # ---------------------------------------------------------------------
        # Team success while players were used
        # ---------------------------------------------------------------------

        F.col("Team_Success_PPM")
        .cast("double")
        .alias("points_per_match"),

        F.col("Team_Success_onG")
        .cast("int")
        .alias("goals_scored"),

        F.col("Team_Success_onGA")
        .cast("int")
        .alias("goals_conceded"),

        F.col("`Team_Success_+/-`")
        .cast("int")
        .alias("goal_difference"),

        F.col("`Team_Success_+/-90`")
        .cast("double")
        .alias("goal_difference_per_90"),


        # ---------------------------------------------------------------------
        # Season / league context
        # ---------------------------------------------------------------------

        F.col("_season")
        .cast("string")
        .alias("season_code"),

        season_code_to_start_year("_season")
        .alias("season_start_year"),

        F.col("_league")
        .cast("string")
        .alias("league_code"),


        # ---------------------------------------------------------------------
        # Bronze lineage metadata
        # ---------------------------------------------------------------------

        F.col("_ingested_at")
        .cast("timestamp")
        .alias("_bronze_ingested_at"),

        F.col("_source_file")
        .cast("string")
        .alias("_bronze_source_file"),

        F.col("_source_system")
        .cast("string")
        .alias("_source_system"),
    )
)


# -----------------------------------------------------------------------------
# Validate business key before deduplication
# -----------------------------------------------------------------------------
# Validate the complete business key before deduplication so rows with NULL
# team, season, or league identifiers cannot be grouped and reduced
# arbitrarily.

validate_required_not_null(
    silver_df,
    BUSINESS_KEY,
    fail_on_null=True,
)


# -----------------------------------------------------------------------------
# Deduplicate repeated Bronze snapshots
# -----------------------------------------------------------------------------
# Bronze currently uses append mode, so rerunning the same FBref Bronze load
# can produce repeated team/season/league records.
#
# Keep the record from the latest Bronze ingestion event.

silver_df = deduplicate_latest(
    silver_df,
    key_columns=BUSINESS_KEY,
    order_column="_bronze_ingested_at",
    tie_breaker_columns=[
        "_bronze_source_file",
    ],
)


# -----------------------------------------------------------------------------
# Resolve canonical team key
# -----------------------------------------------------------------------------

team_mappings_df = load_team_mappings_by_name(
    spark,
    SILVER_SCHEMA,
    "fbref",
)

silver_df = resolve_team_keys_by_name(
    silver_df,
    team_mappings_df,
    source_name_column="team_name",
    output_key_column="team_key",
    fail_on_unmapped=True,
)

print("FBref canonical team resolution completed.")

# -----------------------------------------------------------------------------
# Add Silver processing metadata
# -----------------------------------------------------------------------------

silver_df = silver_df.withColumn(
    "_silver_processed_at",
    F.current_timestamp(),
)


# -----------------------------------------------------------------------------
# Final Silver validation
# -----------------------------------------------------------------------------
# Required analytical identifiers are validated as the final data-quality
# gate.
#
# The helper also returns the final row count, avoiding a separate count()
# action.

validation_metrics = validate_required_not_null(
    silver_df,
    REQUIRED_SILVER_COLUMNS,
    fail_on_null=True,
)


# -----------------------------------------------------------------------------
# Write Silver table
# -----------------------------------------------------------------------------
# The current project phase uses full-refresh Silver processing.

write_full_refresh(
    silver_df,
    SILVER_TABLE,
)


# -----------------------------------------------------------------------------
# Completion message
# -----------------------------------------------------------------------------

print(
    f"Successfully loaded {validation_metrics['row_count']} "
    f"team playing-time records into {SILVER_TABLE}."
)